In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1.0)

producer.flush()
producer.close()

Overwriting producer.py


In [2]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    # Pobieramy dane transakcji ze słownika
    tx = message.value
    
    # if amount > 3000, jeśli tak — wypisz ALERT
    if tx['amount'] > 3000:
        print(f"ALERT: ID: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Overwriting consumer_filter.py


In [3]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Konsument analizujący ryzyko")

for message in consumer:
    tx = message.value
    amount = tx['amount']
    
    # przypisywanie poziomu ryzyka
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    
    # nowe pole do słownika
    tx['risk_level'] = risk_level
    
    # Wyświetlamy wynik z nowym polem
    print(f"[{tx['risk_level']:6}] | ID: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

Overwriting consumer_enrich.py


In [4]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

print("Konsument statystyczny")

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    
    # 1. Zwiększamy licznik transakcji dla sklepu
    store_counts[store] += 1
    
    # 2. Dodajemy kwotę do sumy (inicjalizacja klucza jeśli nie istnieje)
    if store not in total_amount:
        total_amount[store] = 0.0
    total_amount[store] += amount
    
    msg_count += 1
    
    # 3. Co 10 wiadomości tabela podsumowująca
    if msg_count % 10 == 0:
        print(f"\n[ Raport po {msg_count} wiadomościach ]")
        print(f"{'Sklep':<12} | {'Liczba':<7} | {'Suma':<12} | {'Średnia':<10}")
        print("-" * 50)
        
        for s in sorted(store_counts.keys()):
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count
            print(f"{s:<12} | {count:<7} | {total:>9.2f} zł | {avg:>8.2f} zł")
        print("-" * 50)

Overwriting consumer_count.py


In [5]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Inicjalizacja statystyk
category_counts = Counter()
category_total_amount = {}
category_min = {}
category_max = {}
msg_count = 0

print("Konsument statystyczny per produkt")

for message in consumer:
    tx = message.value
    category = tx['category']
    amount = tx['amount']
    
    # 1. Zliczanie
    category_counts[category] += 1
    
    # 2. Inicjalizacja i Sumowanie
    if category not in category_total_amount:
        category_total_amount[category] = 0.0
        category_min[category] = amount
        category_max[category] = amount
    
    category_total_amount[category] += amount
    
    # 3. Aktualizacja MIN i MAX
    if amount < category_min[category]:
        category_min[category] = amount
    if amount > category_max[category]:
        category_max[category] = amount
    
    msg_count += 1
    
    # 4. Tabela co 10 wiadomości
    if msg_count % 10 == 0:
        print(f"\n[ Raport po {msg_count} wiadomościach ]")
        # Nagłówki z nową szerokością:
        # Kategoria (12), Liczba (7), Łączny przychód (18), Min (12), Max (12)
        print(f"{'Kategoria':<12} | {'Liczba':<7} | {'Łączny przychód':<18} | {'Min':<12} | {'Max':<12}")
        print("-" * 75)
        
        for cat in sorted(category_counts.keys()):
            count = category_counts[cat]
            total = category_total_amount[cat]
            c_min = category_min[cat]
            c_max = category_max[cat]
            
            # Formatowanie danych:
            # :>15.2f daje nam wyrównanie kwoty do prawej wewnątrz kolumny 18-znakowej (zostawiając margines na ' zł')
            print(f"{cat:<12} | {count:<7} | {total:>15.2f} zł | {c_min:>9.2f} zł | {c_max:>9.2f} zł")
        print("-" * 75)

Overwriting consumer_stats.py


In [23]:
# Zadanie 5.2 — Pytania
# Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?
# Konsument będzie czekał na nowe wiadomości. Ponieważ producent już skończył pracę, konsument nic nie wyświetli, 
# dopóki nie uruchomimy producenta ponownie.
# Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?

# Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?
# W przetwarzaniu bezstanowym każda wiadomość jest analizowana w izolacji - pozostałe nie są zapisywane i nie mają znaczenia. 
# W przetwarzaniu stanowym wynik zależy od bieżącej i wcześniejszych wiadomości - transkacje muszą być gdzieś zapisywane.

In [6]:
%%file velocity_detector.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Klucz: user_id, wartość: lista krotek (timestamp, amount)
user_history = defaultdict(list)

print("Detektor anamalii prędkości")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    amount = tx['amount']
    current_time = datetime.fromisoformat(tx['timestamp'])
    
    # 1. Dodajemy parę (czas, kwota) do historii
    user_history[user_id].append((current_time, amount))
    
    # 2. Czyścimy starą historię
    user_history[user_id] = [
        t for t in user_history[user_id] 
        if (current_time - t[0]).total_seconds() <= 60
    ]
    
    # 3. Sprawdzamy warunek prędkości
    if len(user_history[user_id]) > 3:
        # Sumujemy drugie elementy (kwoty) z przefiltrowanej listy
        total_velocity_amount = sum(t[1] for t in user_history[user_id])
        
        print(f"ALERT !!! Użytkownik {user_id} wykonał "
              f"{len(user_history[user_id])} transakcje w ciągu 60s! "
              f"Łączna kwota: {total_velocity_amount:.2f} PLN")

Writing velocity_detector.py
